In [20]:
from jinja2 import Template
import pandas as pd
import os
import yaml
import sys
import json
import random
from itertools import product
import torch as t
from transformers import AutoTokenizer, AutoModelForCausalLM
import outlines
from outlines import Generator
from openai import OpenAI
from tqdm import tqdm
sys.path.append("../")
from src.utils import list_to_str, openai_api_call
device = "cpu"

In [3]:
def write_to_json(file, file_path):
    with open(file_path, 'w') as f:
        json.dump(file, f)

def expand_dict_combinations(data):
    keys = list(data.keys())
    values = [data[k] for k in keys]
    
    combos = []
    for prod in product(*values):
        combos.append(dict(zip(keys, prod)))
    return combos

In [4]:
with open('../configs/synthetic_sjt_seeds.yaml', 'r') as file:
    synthetic_sjt_seeds = yaml.safe_load(file)

handmade_sjt_template_df = pd.read_csv("sjt_data/sjt_jinja_template.csv")

In [5]:
all_seed_combos = expand_dict_combinations(synthetic_sjt_seeds)

In [6]:
random.seed(42)
n = 60
sampled_seed_combos =  random.sample(all_seed_combos, n)

In [7]:
SJT_GENERATION_TEMPLATE_STR = """You are creating a new law enforcement Situational Judgment Test (SJT) scenario by modifying an existing scenario with new attribute values. Follow the template below to generate a realistic, professionally appropriate scenario that maintains the core decision-making structure while incorporating the specified attributes.

**Base Scenario to Modify:**
{{base_scenario}}


**Instructions**
* Keep the same personality trait dimensions being tested (Honesty-Humility, Emotionality, Extraversion, Agreeableness, Conscientiousness, Openness to Experience)
* Preserve the professional law enforcement context

**Attribute description**
* **Urgency Level:** Adjust time pressure and decision timeline accordingly
   * Low: Allow deliberation time, non-critical timing
   * Medium: Some time pressure, manageable deadlines
   * High: Immediate decisions required, critical timing
* **Threat Level:** Scale physical danger and safety concerns
   * Low: Administrative issues, minor policy matters
   * Medium: Potential for injury, safety protocols needed
   * High: Life-threatening situations, lethal force considerations
* **Ambiguity Level:** Modify clarity of protocols and guidance
   * Clear: Obvious procedures, straightforward application
   * Moderate: Some judgment required, minor gray areas
   * High: Conflicting guidance, novel situations, ethical dilemmas
* **Individuals Involved:** Adjust scenario complexity
   * Simple: Officer making individual decision
   * Moderate: 2-3 parties with different perspectives
   * Complex: Multiple stakeholders, witnesses, supervisors
* **Authority Relationships:** Frame interactions appropriately
   * Peer Level: Fellow officers, partners, colleagues
   * Subordinate: Supervisors, training officers, senior personnel
   * Authority: Suspects, witnesses, civilians, subordinates
* **Ethical Considerations:** Incorporate specified ethical tension
* **Situation Type:** Adapt setting and context to match type
* **Time of Day:** Include time context naturally in scenario
* **Demographics:** Integrate subject's race, gender, and age naturally without stereotyping if applicable.

**New Attribute Values:**
* **Urgency Level:** {{urgency_level}}
* **Threat Level:** {{threat_level}}
* **Ambiguity Level:** {{ambiguity_level}}
* **Individuals Involved:** {{individuals_involved}}
* **Authority Relationships:** {{authority_relationships}}
* **Ethical Considerations:** {{ethical_considerations}}
* **Situation Type:** {{situation_type}}
* **Time of Day:** {{time_of_day}}
* **Subject Race:** {{race}}
* **Subject Gender:** {{gender}}
* **Subject Age:** {{age}}


**Response Options Requirements**
Generate six response options (1-6) that:
* Clearly differentiate the six personality dimensions
* Remain professionally appropriate and realistic
* Reflect how each personality type would approach the modified scenario
* Maintain consistent quality and plausibility across all options
* The responses are qualitative descriptions of actions and do not contain specific speech suggestions.
* Integrate all specified attributes naturally into the scenario without explicitly naming them
* Do not use direct labels like "high-priority," "mental health crisis," "high threat," etc. in the responses.
* Make attribute levels apparent through context, actions, and circumstances rather than descriptive terms
* Let the urgency, threat level, and situation type emerge from the scenario details rather than being stated outright
* Ensure scenario is realistic and could occur in actual law enforcement
* The scenario should be 2-4 sentences
* Include specific, actionable decisions rather than vague choices
* Verify that all six personality responses are distinct and characteristic

**Output Format**
Provide the complete SJT scenario followed by the six response options in the standard format:

**Scenario:** [Complete scenario description]

**Response Options:**
* 1) Honesty-Humility - [Response and reasoning]
* 2) Emotionality - [Response and reasoning]
* 3) Extraversion - [Response and reasoning]
* 4) Agreeableness - [Response and reasoning]
* 5) Conscientiousness - [Response and reasoning]
* 6) Openness to Experience - [Response and reasoning]
"""

In [9]:
sjt_example_template_str = """

Question: {{ question }}
Options: 

{{ answer_options}}

"""

In [10]:
sjt_example_template = Template(sjt_example_template_str)
SJT_GENERATION_TEMPLATE = Template(SJT_GENERATION_TEMPLATE_STR)

In [11]:
option_cols = ['Option 1', 'Option 2', 'Option 3', 'Option 4', 'Option 5','Option 6']

In [32]:
for index, row in handmade_sjt_template_df.iterrows():
    if index > 3:
        break
    print(index)

0
1
2
3


In [34]:
synthetic_generated_sjt_list = []
for index, row in tqdm(handmade_sjt_template_df.iterrows(), desc = "base_scenario", position=0):
    if index > 3:
        break
    generated_sjt_dict = {}
    question = row['Question']
    answer_options = list_to_str(row[option_cols])
    
    base_scenario = sjt_example_template.render(question = question, answer_options=answer_options)
    for seed_dict in tqdm(random.sample(sampled_seed_combos,5), desc = "seeds", position=1):
        seed_dict['base_scenario'] = base_scenario
        
        sjt_generation_prompt = SJT_GENERATION_TEMPLATE.render(seed_dict)
        generated_sjt_dict['config'] = seed_dict
        openai_sjt_response = openai_api_call(prompt = sjt_generation_prompt, model="gpt-5-mini")
        generated_sjt_dict['generated_sjt'] = openai_sjt_response
        synthetic_generated_sjt_list.append(generated_sjt_dict)

seeds: 100%|██████████| 5/5 [03:00<00:00, 36.17s/it]
base_scenario: 4it [12:07, 181.80s/it]


In [38]:
print(synthetic_generated_sjt_list[0]['generated_sjt'])

**Scenario:** During your morning patrol you stop a middle-aged multiracial woman for a traffic violation. As you approach, she stays partly inside the vehicle with one hand near her jacket pocket and seems unusually agitated; your department's guidance gives different examples about when to remove someone or conduct a search, and a newly introduced field assessment tool is available but its use in this situation isn’t clearly defined. You must decide whether to follow the step-by-step removal/search procedure exactly, call for authorization, or adopt the new tool and alter the sequence to resolve the stop quickly while keeping everyone safe.

**Response Options:**
* 1) Honesty-Humility - You follow the established removal and search procedures exactly: activate your camera, request backup, order the driver out at a safe distance, conduct a pat-down and vehicle check only under the standards set in training, and document every action in the report. You avoid using the new assessment to

In [25]:
openai_sjt_response = openai_api_call(prompt = sjt_generation_prompt, model="gpt-5-mini")

In [27]:
print(openai_sjt_response)

**Scenario:** It’s the morning near the end of your shift when you take an adult assault report; detectives from a neighboring agency arrive saying they need the case file and custody of the subject immediately to pursue a related lead, while your squad supervisor asks you to hand things over quickly so operations aren’t held up. Your department requires officers to complete incident reports before going off duty, and there’s concern the situation could become volatile if critical details are missed during a rushed transfer. You must decide whether to follow the supervisor’s instruction to expedite the handover or to ensure the full documentation is completed first.

**Response Options:**
* 1) Honesty-Humility - Complete the full incident report before any transfer, stay past shift if necessary, and make a contemporaneous note of the supervisor’s request so the decision is transparent; this preserves accuracy and personal accountability even if it creates friction with the team.
* 2) E